<a href="https://colab.research.google.com/github/harshavardhib/Dataviz-Exercises-Harshavardhan-Babu-Bokka/blob/main/lecture07_exerciseipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/content/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1 — Treemap

# 1. Keep fossil-fuel sources only
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# 2. CVD-safe colours for the three fossil sources
#    Keep Coal orange so we can reuse the same colour in Task 3.
fossil_colours = {
    'Coal': '#E07B39',          # orange
    'Oil': '#4C78A8',           # blue
    'Natural Gas': '#F5C842'    # yellow
}

# 3. Find the region with the highest fossil share of its total energy
fossil_by_region = df_fossil.groupby('Region')['TWh'].sum()
total_by_region = df.groupby('Region')['TWh'].sum()
fossil_share_by_region = (fossil_by_region / total_by_region).sort_values(ascending=False)

top_region = fossil_share_by_region.index[0]
top_share = fossil_share_by_region.iloc[0]

# 4. Build the hierarchical treemap: Region → Country → Source
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colours,
    labels={'TWh': 'Energy (TWh)', 'Source': 'Fossil source'},
    title=f'{top_region} has the highest fossil dependency ({top_share:.1%} of its energy)',
    height=750,
    template='plotly_white'
)

# 5. Show TWh values in labels (not percentages)
fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>'
)

# 6. Grey out parent nodes (Region and Country)
fig.data[0].marker.colors = [
    colour if colour in fossil_colours.values() else '#DDDDDD'
    for colour in fig.data[0].marker.colors
]

fig.update_layout(font=dict(family='Arial', size=12))
fig.show()

print(f'Most fossil-dependent region: {top_region} ({top_share:.1%})')

Most fossil-dependent region: Middle East (99.0%)


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
# Task 2 — Treemap and horizontal bar chart

# 1. Keep low-carbon sources only (Nuclear + Hydro)
df_low_carbon = df.loc[df['Source_Type'] == 'Low-carbon'].copy()

# 2. Aggregate TWh by country
low_carbon_by_country = (
    df_low_carbon.groupby('Country', as_index=False)['TWh']
    .sum()
    .sort_values('TWh', ascending=False)
)

# 3. Add a dummy root column for the treemap
low_carbon_by_country['All'] = 'Low-carbon'

# 4. Treemap — one hierarchy level under the dummy root
fig_treemap = px.treemap(
    low_carbon_by_country,
    path=['All', 'Country'],
    values='TWh',
    color_discrete_sequence=['#0072B2'],
    labels={'TWh': 'Energy (TWh)'},
    title='Low-carbon energy by country',
    height=650,
    template='plotly_white'
)

fig_treemap.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>'
)
fig_treemap.show()

# 5. Horizontal bar chart — countries sorted from highest to lowest
leading_country = low_carbon_by_country.iloc[0]['Country']
leading_twh = low_carbon_by_country.iloc[0]['TWh']

fig_bar = px.bar(
    low_carbon_by_country,
    x='TWh',
    y='Country',
    orientation='h',
    color_discrete_sequence=['#0072B2'],
    labels={'TWh': 'Energy (TWh)', 'Country': ''},
    title=f'{leading_country} leads in low-carbon energy ({leading_twh:,.0f} TWh)',
    height=600,
    template='plotly_white'
)

fig_bar.update_traces(
    texttemplate='%{x:,.0f} TWh',
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} TWh<extra></extra>'
)
fig_bar.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_bar.show()

print(f'Leading country: {leading_country} ({leading_twh:,.0f} TWh)')

Leading country: France (7,593 TWh)


### Task 2 — reflection

The **horizontal bar chart** tells the ranking story more clearly because all bars share the same starting point, so it is easier to compare lengths and see which country has more low-carbon energy. The treemap is useful for showing the relative size of each country's contribution, but comparing rectangle areas is less precise.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [4]:
# Task 3 — 100% stacked bar chart

# 1. Aggregate fossil TWh by Region and Source
region_source = (
    df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh']
    .sum()
)

# 2. Convert each source value into a share of its region's fossil total
region_source['Share'] = (
    region_source['TWh']
    / region_source.groupby('Region')['TWh'].transform('sum')
)

# 3. Sort regions by coal share, highest first
coal_share_by_region = (
    region_source.loc[region_source['Source'] == 'Coal']
    .set_index('Region')['Share']
    .sort_values(ascending=False)
)
region_order = coal_share_by_region.index.tolist()

# 4. Add percentage labels only to the Coal segments
region_source['Label'] = np.where(
    region_source['Source'] == 'Coal',
    region_source['Share'].map('{:.0%}'.format),
    ''
)

# 5. Use the same orange for Coal as in Task 1; Oil and Natural Gas are grey
fossil_bar_colours = {
    'Coal': fossil_colours['Coal'],
    'Oil': '#BBBBBB',
    'Natural Gas': '#DDDDDD'
}

top_coal_region = coal_share_by_region.index[0]
top_coal_share = coal_share_by_region.iloc[0]

# 6. Build the horizontal 100% stacked bar chart
fig = px.bar(
    region_source,
    x='Share',
    y='Region',
    color='Source',
    text='Label',
    orientation='h',
    color_discrete_map=fossil_bar_colours,
    category_orders={
        'Region': region_order,
        'Source': ['Coal', 'Oil', 'Natural Gas']
    },
    labels={'Share': 'Share of fossil energy', 'Source': 'Fossil source'},
    title=f'{top_coal_region} relies most on coal: {top_coal_share:.1%} of its fossil energy',
    height=550,
    template='plotly_white'
)

fig.update_traces(
    textposition='inside',
    insidetextanchor='start',
    textfont=dict(color='white', size=11),
    marker_line_color='white',
    marker_line_width=1,
    hovertemplate='<b>%{y}</b><br>%{fullData.name}: %{x:.1%}<extra></extra>'
)

fig.update_layout(
    xaxis=dict(tickformat='.0%', range=[0, 1], title='Share of fossil energy'),
    yaxis=dict(title=''),
    legend=dict(orientation='h', x=0, y=1.12),
    font=dict(family='Arial', size=12),
    bargap=0.25
)

fig.show()

print(f'Most coal-dependent region: {top_coal_region} ({top_coal_share:.1%} of fossil energy)')

Most coal-dependent region: Africa (78.0% of fossil energy)


### Task 3 — reflection

The **Task 1 treemap** answers: “How much fossil energy comes from each region, country and source?” It shows the hierarchy and the relative TWh magnitude.

The **100% stacked bar chart** answers: “Which region has the highest coal share within its fossil energy mix?” It makes the composition easier to compare because every region's bar has the same 100% length.